# Unit Test Generator + Model Benchmark

Week 4 community contribution: generate deterministic pytest tests from pasted Python code, run tests, and benchmark models.

In [ ]:
# Install dependencies if needed (rerun-safe)
# If your repo uses uv, you can alternatively run: uv add pytest pandas gradio openai python-dotenv
%pip -q install pytest pandas gradio openai python-dotenv

In [ ]:
from __future__ import annotations

import os
import re
import subprocess
import time
from pathlib import Path

import gradio as gr
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:


load_dotenv()

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "week4").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd())
BASE_DIR = REPO_ROOT / "week4" / "community-contributions" / "CynthiaOmovoIye"
GENERATED_CODE_DIR = BASE_DIR / "generated_code"
GENERATED_TESTS_DIR = BASE_DIR / "generated_tests"
TARGET_PATH = GENERATED_CODE_DIR / "target.py"
TEST_PATH = GENERATED_TESTS_DIR / "test_generated.py"

# Week 4 style: list + map to OpenAI-compatible clients
openai_client = OpenAI()
anthropic = OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1/")
gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
grok = OpenAI(api_key=os.getenv("GROK_API_KEY"), base_url="https://api.x.ai/v1")
groq = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
openrouter = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

models = [
    "gpt-5",
    "claude-sonnet-4-5-20250929",
    "grok-4",
    "gemini-2.5-pro",
    "qwen2.5-coder",
    "deepseek-coder-v2",
    "gpt-oss:20b",
    "qwen/qwen3-coder-30b-a3b-instruct",
    "openai/gpt-oss-120b",
]

model_to_client = {
    "gpt-5": openai_client,
    "claude-sonnet-4-5-20250929": anthropic,
    "grok-4": grok,
    "gemini-2.5-pro": gemini,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "gpt-oss:20b": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

BASE_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_TESTS_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """You generate Python unit tests.
Return ONLY valid pytest code.
No markdown. No triple backticks. No explanations."""

USER_PROMPT_TEMPLATE = """Write deterministic pytest tests for the Python code below.

Rules:
- Assume the code is saved at: generated_code/target.py
- Import it using safe import-by-path (do NOT rely on sys.path):
  Use importlib.util + pathlib to load target.py as a module named \"target\".
- If the code defines functions/classes, test them directly.
- If the code is a script, test that it runs without crashing and (if it prints) assert key output patterns using capsys.
- Include:
  1) happy path test
  2) edge case test
  3) regression-style test
- No flaky tests, no network, no randomness unless seeded.
- Keep tests small and fast.

Here is the code:
<<<PYTHON
{python_code}
PYTHON

If you need to \"wrap\" script code, do it in the tests without editing the original code.
Return ONLY pytest code."""

RETRY_INSTRUCTION = "Your previous tests failed. Fix imports and references. Do not change target code. Return only pytest code."

def _extract_text_content(response) -> str:
    text = getattr(response.choices[0].message, "content", "")
    if isinstance(text, list):
        chunks = []
        for item in text:
            if isinstance(item, dict):
                if item.get("type") in {"text", "output_text"} and item.get("text"):
                    chunks.append(item["text"])
            elif hasattr(item, "text"):
                chunks.append(item.text)
        text = "\n".join(chunks)
    return str(text or "").strip()

def _strip_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(?:python)?\\s*", "", text)
    text = re.sub(r"\\s*```$", "", text)
    return text.strip()

def _normalize_generated_tests_code(test_code: str) -> str:
    """Fix common wrong target.py path patterns in model output."""
    fixed = test_code
    fixed = fixed.replace("generated_tests/generated_code/target.py", "generated_code/target.py")
    fixed = fixed.replace("generated_tests\\\\generated_code\\\\target.py", "generated_code\\\\target.py")
    fixed = fixed.replace("Path(__file__).parent / \"generated_code\"", "Path(__file__).resolve().parents[1] / \"generated_code\"")
    fixed = fixed.replace("Path(__file__).resolve().parent / \"generated_code\"", "Path(__file__).resolve().parents[1] / \"generated_code\"")
    fixed = fixed.replace("Path(__file__).resolve().parent / 'generated_code'", "Path(__file__).resolve().parents[1] / 'generated_code'")
    fixed = fixed.replace("Path(__file__).parent / 'generated_code'", "Path(__file__).resolve().parents[1] / 'generated_code'")
    return fixed

def _strip_ansi(text: str) -> str:
    return re.sub(r"\\x1b\[[0-9;]*m", "", text or "")

def _pytest_summary(output: str, exit_code: int) -> str:
    output = _strip_ansi(output)
    summary_line = ""
    for line in output.splitlines()[::-1]:
        if "passed" in line or "failed" in line or "error" in line.lower():
            summary_line = line.strip()
            break
    if not summary_line:
        summary_line = (output.strip().splitlines()[-1] if output.strip() else "No pytest output")
    status = "PASS" if exit_code == 0 else "FAIL"
    return f"{status} | {summary_line}"[:240]

def _format_run_output(output: str, exit_code: int, seconds: float, retry_msg: str) -> str:
    clean = _strip_ansi(output).strip()
    summary = _pytest_summary(clean, exit_code)
    interesting = []
    for line in clean.splitlines():
        lower = line.lower()
        if any(k in lower for k in ["error", "failed", "traceback", "assert", "passed"]):
            interesting.append(line)
    interesting = interesting[:12]
    details = "\n".join(interesting) if interesting else (clean[:1500] if clean else "No pytest output.")
    status_label = "PASSED" if exit_code == 0 else "FAILED"
    return (
        f"Test Result: {status_label}\n"
        f"Exit code: {exit_code}\n"
        f"Pytest time: {seconds:.3f}s\n"
        f"Retry: {retry_msg}\n"
        f"Summary: {summary}\n\n"
        f"Key pytest lines:\n{details}"
    )


In [ ]:
def write_python_target(python_code: str) -> str:
    """Writes python_code to generated_code/target.py and returns the path."""
    GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)
    TARGET_PATH.write_text(python_code or "", encoding="utf-8")
    return str(TARGET_PATH)


def _messages_for_tests(python_code: str, retry: bool = False) -> list[dict[str, str]]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(python_code=python_code)},
    ]
    if retry:
        messages.append({"role": "user", "content": RETRY_INSTRUCTION})
    return messages


def generate_tests(model: str, python_code: str) -> str:
    """Calls the LLM, returns pytest code string, and writes generated_tests/test_generated.py."""
    try:
        write_python_target(python_code)
        GENERATED_TESTS_DIR.mkdir(parents=True, exist_ok=True)
        client = model_to_client[model]
        kwargs = {
            "model": model,
            "messages": _messages_for_tests(python_code, retry=False),
        }
        if "gpt" in model:
            kwargs["reasoning_effort"] = "medium"
        response = client.chat.completions.create(**kwargs)
        test_code = _normalize_generated_tests_code(_strip_fences(_extract_text_content(response)))
        TEST_PATH.write_text(test_code, encoding="utf-8")
        return test_code
    except Exception as e:
        err = f"# Generation error\n# {type(e).__name__}: {e}\n"
        TEST_PATH.write_text(err, encoding="utf-8")
        return err


def _generate_tests_with_retry(model: str, python_code: str) -> str:
    """Internal helper: retry once with import/reference-fix instruction."""
    client = model_to_client[model]
    kwargs = {
        "model": model,
        "messages": _messages_for_tests(python_code, retry=True),
    }
    if "gpt" in model:
        kwargs["reasoning_effort"] = "medium"
    response = client.chat.completions.create(**kwargs)
    test_code = _normalize_generated_tests_code(_strip_fences(_extract_text_content(response)))
    TEST_PATH.write_text(test_code, encoding="utf-8")
    return test_code


def run_tests() -> tuple[str, int, float]:
    """Runs pytest -q generated_tests/test_generated.py and returns (output, exit_code, seconds)."""
    GENERATED_TESTS_DIR.mkdir(parents=True, exist_ok=True)
    cmd = ["pytest", "-q", "generated_tests/test_generated.py"]
    start = time.perf_counter()
    try:
        completed = subprocess.run(cmd, cwd=str(BASE_DIR), text=True, capture_output=True, check=False)
        seconds = time.perf_counter() - start
        output = (completed.stdout or "") + ("\n" + completed.stderr if completed.stderr else "")
        return output.strip(), int(completed.returncode), float(seconds)
    except Exception as e:
        seconds = time.perf_counter() - start
        return f"Error while running pytest: {type(e).__name__}: {e}", 1, float(seconds)


def evaluate_models_for_tests(models: list[str], python_code: str, runs: int = 1) -> pd.DataFrame:
    """Benchmark test generation + pytest execution for each model."""
    safe_runs = max(1, int(runs or 1))
    rows = []

    for model in models:
        for _ in range(safe_runs):
            gen_start = time.perf_counter()
            _ = generate_tests(model, python_code)
            gen_seconds = time.perf_counter() - gen_start

            pytest_output, exit_code, pytest_seconds = run_tests()

            # One automatic retry only for likely import/reference issues.
            if exit_code != 0 and re.search(r"(ImportError|ModuleNotFoundError|NameError|cannot import name|FileNotFoundError|No such file or directory)", pytest_output):
                try:
                    retry_start = time.perf_counter()
                    _ = _generate_tests_with_retry(model, python_code)
                    gen_seconds += time.perf_counter() - retry_start
                    pytest_output, exit_code, pytest_seconds = run_tests()
                except Exception as retry_error:
                    pytest_output = f"Retry generation failed: {type(retry_error).__name__}: {retry_error}\n\n{pytest_output}"
                    exit_code = 1

            rows.append(
                {
                    "model": model,
                    "gen_seconds": round(float(gen_seconds), 4),
                    "pytest_seconds": round(float(pytest_seconds), 4),
                    "exit_code": int(exit_code),
                    "passed": bool(exit_code == 0),
                    "summary": _pytest_summary(pytest_output, exit_code),
                }
            )

    df = pd.DataFrame(rows, columns=["model", "gen_seconds", "pytest_seconds", "exit_code", "passed", "summary"])
    if not df.empty:
        display(df)
    return df


In [ ]:
def _run_tests_with_retry(model: str, python_code: str) -> tuple[str, int, float, str]:
    output, exit_code, seconds = run_tests()
    retried = False
    if exit_code != 0 and re.search(r"(ImportError|ModuleNotFoundError|NameError|cannot import name|FileNotFoundError|No such file or directory)", output):
        retried = True
        try:
            _generate_tests_with_retry(model, python_code)
            output, exit_code, seconds = run_tests()
        except Exception as e:
            output = f"Retry failed: {type(e).__name__}: {e}\n\n{output}"
            exit_code = 1
    msg = "Retried once for import/reference issue." if retried else "No retry needed."
    return output, exit_code, seconds, msg


def ui_generate_tests(model: str, python_code: str) -> tuple[str, str]:
    try:
        tests = generate_tests(model, python_code)
        return tests, "Tests generated and saved to generated_tests/test_generated.py"
    except Exception as e:
        return "", f"Generation failed: {type(e).__name__}: {e}"


def ui_run_tests(model: str, python_code: str, current_tests: str) -> tuple[str, str, str]:
    try:
        if python_code and not TARGET_PATH.exists():
            write_python_target(python_code)
        if current_tests and not TEST_PATH.exists():
            TEST_PATH.write_text(current_tests, encoding="utf-8")
        output, exit_code, seconds, retry_msg = _run_tests_with_retry(model, python_code)
        formatter = globals().get("_format_run_output")
        if callable(formatter):
            pretty_output = formatter(output, exit_code, seconds, retry_msg)
        else:
            clean = re.sub(r"\\x1b\[[0-9;]*m", "", output or "").strip()
            pretty_output = (
                f"Test Result: {('PASSED' if exit_code == 0 else 'FAILED')}\n"
                f"Exit code: {exit_code}\n"
                f"Pytest time: {seconds:.3f}s\n"
                f"Retry: {retry_msg}\n\n"
                f"Output:\n{clean}"
            )
        status = f"{('PASS' if exit_code == 0 else 'FAIL')} | {seconds:.3f}s"
        return TEST_PATH.read_text(encoding="utf-8"), pretty_output, status
    except Exception as e:
        return current_tests or "", "", f"Run failed: {type(e).__name__}: {e}"


def ui_benchmark(python_code: str, selected_models: list[str], runs: int) -> tuple[pd.DataFrame, str]:
    try:
        chosen = selected_models or [models[0]]
        df = evaluate_models_for_tests(chosen, python_code, runs=runs)
        if df.empty:
            return df, "No benchmark rows generated."
        pass_rate = float(df["passed"].mean() * 100.0)
        best_row = df.sort_values(["exit_code", "gen_seconds", "pytest_seconds"]).iloc[0]
        summary = (
            f"Benchmarked {len(df)} run(s) across {len(chosen)} model(s). "
            f"Pass rate: {pass_rate:.1f}%. "
            f"Best row: {best_row['model']} | gen={best_row['gen_seconds']}s | pytest={best_row['pytest_seconds']}s | exit={best_row['exit_code']}."
        )
        return df, summary
    except Exception as e:
        return pd.DataFrame(), f"Benchmark failed: {type(e).__name__}: {e}"


In [ ]:
sample_python = """
def divide(a, b):
    if b == 0:
        raise ValueError('b must not be zero')
    return a / b

class Greeter:
    def hello(self, name: str) -> str:
        return f'Hello, {name}!'
""".strip()

with gr.Blocks() as ui:
    gr.Markdown("## Unit Test Generator Benchmark (Week 4)")
    gr.Markdown(
        "Use **Selected model** for one-off generation/testing. "
        "Use **Benchmark models** to compare multiple models side-by-side."
    )
    with gr.Row():
        python_input = gr.Textbox(label="Python code", lines=24, value=sample_python)
        generated_tests_panel = gr.Code(label="Generated tests", language="python", lines=24)

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=models,
            value=models[0],
            label="Selected model (Generate/Run)",
            info="Used by 'Generate Tests' and 'Run Tests'."
        )

    with gr.Row():
        btn_generate = gr.Button("Generate Tests (Selected Model)", variant="primary")
        btn_run = gr.Button("Run Tests (Current Generated File)")

    with gr.Accordion("Model Comparison", open=False):
        gr.Markdown("Select one or more models below, then run benchmark to compare speed and pass rate.")
        with gr.Row():
            benchmark_models = gr.CheckboxGroup(
                choices=models,
                value=[models[0]],
                label="Benchmark models (compare many)",
                info="Used only by 'Benchmark Models'. Select 1 or more models."
            )
            runs_input = gr.Number(value=1, precision=0, label="Runs per model")
        btn_benchmark = gr.Button("Benchmark Models (Compare Selected)")
        benchmark_df = gr.Dataframe(label="Benchmark table")
        benchmark_summary = gr.Textbox(label="Benchmark summary", lines=3)

    generation_status = gr.Textbox(label="Generation status", lines=2)
    pytest_output_panel = gr.Textbox(label="Pytest output", lines=12)
    run_status = gr.Textbox(label="Run status", lines=2)

    btn_generate.click(
        ui_generate_tests,
        inputs=[model_dropdown, python_input],
        outputs=[generated_tests_panel, generation_status],
    )
    btn_run.click(
        ui_run_tests,
        inputs=[model_dropdown, python_input, generated_tests_panel],
        outputs=[generated_tests_panel, pytest_output_panel, run_status],
    )
    btn_benchmark.click(
        ui_benchmark,
        inputs=[python_input, benchmark_models, runs_input],
        outputs=[benchmark_df, benchmark_summary],
    )

ui.launch(inbrowser=True)

## Demo 1: Paste sample python code -> generate tests -> run pytest

In [ ]:
demo_code = """
def normalize(values):
    if not values:
        return []
    total = sum(values)
    return [v / total for v in values]
""".strip()

selected_model = models[0]
generated = generate_tests(selected_model, demo_code)
print("Generated tests length:", len(generated))
test_output, test_exit, test_secs = run_tests()
print(f"exit={test_exit} pytest_seconds={test_secs:.3f}")
print(test_output[:2000])

## Demo 2: Benchmark across selected models

In [ ]:
benchmark_models_demo = [m for m in models if m in model_to_client][:3]
benchmark_df_demo = evaluate_models_for_tests(benchmark_models_demo, demo_code, runs=1)
benchmark_df_demo